# EmpowerLens — Combined (Annotated + CODIPAS) classification Kaggle GPU runner

Fine-tunes MentalBERT for whole-message classification (binary / multiclass / multilabel) on the **combined** train set (Annotated_data train + CODIPAS-cls train, concatenated/shuffled), evaluated on the frozen Annotated_data val/test — reusing `train_transformer.py` / `evaluate.py` completely unmodified. Same task as your original Annotated_data.csv and CODIPAS-only runs, just a bigger training set.

**Before running:**
1. Settings -> **Accelerator: GPU**, **Internet: On**.
2. `data/splits_combined/{train,val,test}.csv` + `split_manifest.json` must already be **committed and pushed** on the branch below (run `src/make_splits_combined.py` locally first, then commit).
3. `HF_TOKEN` secret must be set (same one from your earlier MentalBERT run).

In [ ]:
# 1. Clone the repo and install the transformer stack.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt

In [ ]:
# 1b. MentalBERT is gated -- log in with the HF_TOKEN secret (Add-ons -> Secrets).
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [ ]:
# 2. Choose ONE task, train MentalBERT over 3 seeds on the COMBINED split, evaluate each.
#    Same train_transformer.py / evaluate.py as before -- only --splits/--out differ,
#    so this never collides with your Annotated_data.csv or CODIPAS-only checkpoints/results.
TASK  = "multiclass"  # one of: binary | multiclass | multilabel
MODEL = "mental/mental-bert-base-uncased"
SPLITS = "data/splits_combined"

for seed in (42, 1337, 2024):
    ckpt = f"checkpoints_combined/{TASK}_{MODEL.split('/')[-1]}_{seed}"
    !python -m src.train_transformer --task $TASK --model $MODEL --seed $seed --device auto --splits $SPLITS --out checkpoints_combined
    !python -m src.evaluate --checkpoint $ckpt --splits $SPLITS --out results_combined --reference

In [ ]:
# 3. Copy results_combined/ to Kaggle output, then prune the heavy stuff
#    (checkpoints_combined, .git) so the Output tab stays small and downloadable.
!mkdir -p /kaggle/working/results_combined
!cp -r results_combined/* /kaggle/working/results_combined/
!rm -rf /kaggle/working/empowerlens/checkpoints_combined
!rm -rf /kaggle/working/empowerlens/.git
!ls -la /kaggle/working/results_combined